In [5]:
# file: preprocess_final_variant_scores.py
import pandas as pd

# 1. Load the dataset
# Adjust delimiter if needed (some .txt files are actually TSV)
df = pd.read_csv("../../data/external/final_variant_scores.txt")

# Inspect the columns (uncomment if you want to check)
# print(df.columns)

# 2. Keep only Wuhan RBD (target == 'Wuhan-Hu-1' and positions 331–531)
df_rbd = df[
    (df["target"] == "Wuhan-Hu-1") &
    (df["position"].between(331, 531))
].copy()

# 3. Drop rows where mutant is stop codon (*)
df_rbd = df_rbd[df_rbd["mutant"] != "*"]

# 4. Keep only useful columns
df_clean = df_rbd[[
    "wildtype", "position", "mutant", "mutation",
    "bind", "delta_bind",
    "expr", "delta_expr"
]].copy()

# 5. Handle missing values (drop NaNs for ML)
df_clean = df_clean.dropna(subset=["delta_bind", "delta_expr"])

# 6. Save the clean dataset
df_clean.to_csv("../../data/processed/rbd_single_mutations_clean.csv", index=False)

print(f"Cleaned dataset saved to rbd_single_mutations_clean.csv")
print(df_clean.head())


Cleaned dataset saved to rbd_single_mutations_clean.csv
      wildtype  position mutant mutation     bind  delta_bind      expr  \
12060        N       331      A    N331A  8.79360     0.06027  10.29895   
12061        N       331      C    N331C  8.61594    -0.15567   9.67665   
12062        N       331      D    N331D  8.75409    -0.01751  10.06985   
12063        N       331      E    N331E  8.92561     0.15400  10.18436   
12064        N       331      F    N331F  8.65690    -0.11470  10.01397   

       delta_expr  
12060     0.11422  
12061    -0.50923  
12062    -0.11602  
12063    -0.00151  
12064    -0.17191  


In [3]:
# file: analyze_rbd_mutations.py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import mean_squared_error, r2_score, classification_report

# 1. Load cleaned dataset
df = pd.read_csv("../../data/processed/rbd_single_mutations_clean.csv")
print("Data shape:", df.shape)
print(df.head())

# -------------------------------
# 2. EDA (Exploratory Data Analysis)
# -------------------------------

# Histogram of delta_bind
plt.figure(figsize=(6,4))
sns.histplot(df['delta_bind'], bins=40, kde=True)
plt.title("Distribution of ΔBind")
plt.xlabel("ΔBind (mutation - WT)")
plt.savefig("hist_delta_bind.png", dpi=150)
plt.close()

# Histogram of delta_expr
plt.figure(figsize=(6,4))
sns.histplot(df['delta_expr'], bins=40, kde=True, color="orange")
plt.title("Distribution of ΔExpr")
plt.xlabel("ΔExpr (mutation - WT)")
plt.savefig("hist_delta_expr.png", dpi=150)
plt.close()

# Scatter ΔBind vs ΔExpr
plt.figure(figsize=(6,6))
sns.scatterplot(x="delta_bind", y="delta_expr", data=df, alpha=0.5)
plt.axhline(0, ls="--", color="gray")
plt.axvline(0, ls="--", color="gray")
plt.title("ΔBind vs ΔExpr")
plt.savefig("scatter_bind_vs_expr.png", dpi=150)
plt.close()

print("EDA plots saved (hist_delta_bind.png, hist_delta_expr.png, scatter_bind_vs_expr.png)")

# -------------------------------
# 3. Feature Encoding
# -------------------------------

# For baseline: features = wildtype, mutant, position
X = df[["wildtype", "mutant", "position"]]
y_reg = df["delta_bind"]   # regression target
# For classification: beneficial / neutral / deleterious from delta_bind
def label_effect(x, thresh=0.2):
    if x > thresh:
        return "beneficial"
    elif x < -thresh:
        return "deleterious"
    else:
        return "neutral"

y_clf = df["delta_bind"].apply(label_effect)

# Column transformer: one-hot for wildtype/mutant, scale position
preprocessor = ColumnTransformer(
    transformers=[
        ("onehot", OneHotEncoder(handle_unknown="ignore"), ["wildtype", "mutant"]),
        ("scaler", StandardScaler(), ["position"])
    ]
)

# -------------------------------
# 4. Regression model (ΔBind)
# -------------------------------
X_train, X_test, y_train, y_test = train_test_split(X, y_reg, test_size=0.2, random_state=42)

reg_model = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("regressor", LinearRegression())
])

reg_model.fit(X_train, y_train)
y_pred = reg_model.predict(X_test)

print("Regression performance:")
print("  RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))
print("  R2:", r2_score(y_test, y_pred))

# -------------------------------
# 5. Classification model
# -------------------------------
X_train, X_test, y_train, y_test = train_test_split(X, y_clf, test_size=0.2, random_state=42, stratify=y_clf)

clf_model = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("classifier", LogisticRegression(max_iter=200))
])

clf_model.fit(X_train, y_train)
y_pred = clf_model.predict(X_test)

print("\nClassification performance:")
print(classification_report(y_test, y_pred))


Data shape: (3899, 8)
  wildtype  position mutant mutation     bind  delta_bind      expr  \
0        N       331      A    N331A  8.79360     0.06027  10.29895   
1        N       331      C    N331C  8.61594    -0.15567   9.67665   
2        N       331      D    N331D  8.75409    -0.01751  10.06985   
3        N       331      E    N331E  8.92561     0.15400  10.18436   
4        N       331      F    N331F  8.65690    -0.11470  10.01397   

   delta_expr  
0     0.11422  
1    -0.50923  
2    -0.11602  
3    -0.00151  
4    -0.17191  
EDA plots saved (hist_delta_bind.png, hist_delta_expr.png, scatter_bind_vs_expr.png)
Regression performance:
  RMSE: 1.0504454384801247
  R2: 0.1634292911338775

Classification performance:
              precision    recall  f1-score   support

  beneficial       0.00      0.00      0.00        11
 deleterious       0.70      0.89      0.78       506
     neutral       0.54      0.28      0.37       263

    accuracy                           0.67    

C:\Users\avant\AppData\Roaming\Python\Python313\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\avant\AppData\Roaming\Python\Python313\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\avant\AppData\Roaming\Python\Python313\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} i